In [1]:
from pathlib import Path
import sys

import pandas as pd
import networkx as nx

PROJECT_ROOT = Path("..")
SRC_DIR = PROJECT_ROOT / "src"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

sys.path.append(str(SRC_DIR))

from train_node2vec import (
    load_graph_from_csv,
    train_node2vec,
    extract_node_embeddings,
    get_nodes_by_type,
    get_node_metadata,
    save_embeddings
)


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nodes_dp_path = PROCESSED_DIR / "nodes_disease_phenotype.csv"
edges_dp_path = PROCESSED_DIR / "edges_disease_phenotype.csv"

G_dp = load_graph_from_csv(
    nodes_path=nodes_dp_path,
    edges_path=edges_dp_path,
)

print("Graph A: disease-phenotype only")
print("Nodes:", G_dp.number_of_nodes())
print("Edges:", G_dp.number_of_edges())

node_types = pd.Series(
    [data.get("node_type") for _, data in G_dp.nodes(data=True)]
).value_counts()

node_types

Graph A: disease-phenotype only
Nodes: 5569
Edges: 20132


phenotype    4569
disease      1000
Name: count, dtype: int64

In [3]:
nodes_dph_path = PROCESSED_DIR / "nodes_disease_phenotype_hpo_hierarchy.csv"
edges_dph_path = PROCESSED_DIR / "edges_disease_phenotype_hpo_hierarchy.csv"

G_dph = load_graph_from_csv(
    nodes_path=nodes_dph_path,
    edges_path=edges_dph_path,
)

print("Graph B: disease-phenotype + HPO hierarchy")
print("Nodes:", G_dph.number_of_nodes())
print("Edges:", G_dph.number_of_edges())

node_types = pd.Series(
    [data.get("node_type") for _, data in G_dph.nodes(data=True)]
).value_counts()

node_types



Graph B: disease-phenotype + HPO hierarchy
Nodes: 11760
Edges: 31521


phenotype    10760
disease       1000
Name: count, dtype: int64

In [4]:
DIMENSIONS = 32

model_dp = train_node2vec(
    graph=G_dp,
    dimensions=DIMENSIONS,
    walk_length=20,
    num_walks=20,
    window=5,
    workers=4,
    seed=5,
)

Computing transition probabilities:   0%|          | 0/5569 [00:00<?, ?it/s]

Generating walks (CPU: 4): 100%|██████████| 5/5 [00:06<00:00,  1.28s/it]


In [5]:
disease_nodes_dp = get_nodes_by_type(G_dp, "disease")

disease_embeddings_dp = extract_node_embeddings(
    model=model_dp,
    nodes=disease_nodes_dp,
    dimensions=DIMENSIONS,
)

nodes_metadata_dp = get_node_metadata(G_dp)

disease_embeddings_dp = disease_embeddings_dp.merge(
    nodes_metadata_dp,
    on="node_id",
    how="left",
)

disease_embeddings_dp.head()


,node_id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_24,dim_25,dim_26,dim_27,dim_28,dim_29,dim_30,dim_31,node_type,label
0,OMIM:619340,-1.299829,-0.655932,0.163469,-0.212847,-0.541585,0.235768,-0.439621,0.706993,-0.094053,...,0.821837,0.886023,0.042071,-0.599112,-0.381867,-0.182142,0.181751,-0.449084,disease,Developmental and epileptic encephalopathy 96
1,OMIM:619426,-0.326325,-0.137298,-0.193332,-0.442689,0.837968,-0.773861,0.525760,-0.003559,0.135980,...,0.710985,0.379728,-0.270676,-0.485451,0.311005,-0.033072,-0.061557,0.322869,disease,White-Kernohan syndrome
2,OMIM:137580,-0.050143,-0.167317,0.572092,-0.152065,0.805713,-0.183667,-0.217657,0.685667,1.586034,...,0.554641,0.784594,0.590065,-0.747803,-0.186101,-0.170492,-0.984656,-0.651949,disease,Gilles de la tourette syndrome
3,OMIM:108770,-0.231050,-0.012970,-0.462338,-0.735995,0.797530,-0.136100,-0.248430,0.346945,-1.308965,...,-0.796522,1.766711,0.500811,-1.599624,-1.881484,1.273753,0.729441,-1.083026,disease,Atrial standstill 1
4,OMIM:615234,0.262825,-0.077044,-0.354960,0.172508,-0.383768,-0.116940,-0.284227,-0.188844,1.058862,...,-0.234586,1.020362,-0.052442,-0.932727,-0.214134,0.556940,0.323526,0.316028,disease,"Anemia, hypochromic microcytic, with iron over..."


In [6]:
save_embeddings(
    disease_embeddings_dp,
    PROCESSED_DIR / "disease_embeddings_node2vec_dp.csv"
)

print("Saved:", PROCESSED_DIR / "disease_embeddings_node2vec_dp.csv")

Saved: ../data/processed/disease_embeddings_node2vec_dp.csv


In [7]:
model_dph = train_node2vec(
    graph=G_dph,
    dimensions=DIMENSIONS,
    walk_length=20,
    num_walks=20,
    window=5,
    workers=4,
    seed=5,
)

Generating walks (CPU: 4): 100%|██████████| 5/5 [00:20<00:00,  4.06s/it]


In [8]:
disease_nodes_dph = get_nodes_by_type(G_dph, "disease")

disease_embeddings_dph = extract_node_embeddings(
    model=model_dph,
    nodes=disease_nodes_dph,
    dimensions=DIMENSIONS,
)

node_metadata_dph = get_node_metadata(G_dph)

disease_embeddings_dph = disease_embeddings_dph.merge(
    node_metadata_dph,
    on="node_id",
    how="left",
)

disease_embeddings_dph.head()

,node_id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_24,dim_25,dim_26,dim_27,dim_28,dim_29,dim_30,dim_31,node_type,label
0,OMIM:619340,-0.387484,-0.910157,0.982619,0.607956,0.958315,-0.368261,0.394461,0.445716,-0.423625,...,-0.120905,0.763132,-0.503885,-0.006861,-0.187148,0.956883,1.158932,0.808067,disease,Developmental and epileptic encephalopathy 96
1,OMIM:619426,-0.767676,-0.366015,0.517694,0.434301,0.294414,-0.705663,-0.861528,0.563413,-0.096870,...,0.735737,0.756306,-0.449358,-0.476867,0.487929,-0.073868,0.637030,-0.079910,disease,White-Kernohan syndrome
2,OMIM:137580,0.327134,0.499508,1.228294,-0.119647,1.322186,-1.136861,-0.067582,0.030970,0.003284,...,0.897514,1.406338,0.542065,-0.363319,-0.000300,-0.003450,0.162700,0.031030,disease,Gilles de la tourette syndrome
3,OMIM:108770,0.965642,-1.905361,0.444665,0.971482,1.557175,-0.560559,0.677274,-0.461704,0.438903,...,1.009293,0.851138,-0.061887,0.333160,-0.838361,-0.896541,1.307505,0.096471,disease,Atrial standstill 1
4,OMIM:615234,-0.508136,-0.921777,0.294369,0.567672,1.090082,-0.122706,-0.327817,0.627031,0.638268,...,-0.244192,-0.393572,-1.116120,-0.545966,-1.253565,0.190854,0.288518,-0.912181,disease,"Anemia, hypochromic microcytic, with iron over..."


In [10]:
save_embeddings(
    disease_embeddings_dph,
    PROCESSED_DIR / "disease_embeddings_node2vec_dph.csv"
)

print("Saved:", PROCESSED_DIR / "disease_embeddings_node2vex_dph.csv")

Saved: ../data/processed/disease_embeddings_node2vex_dph.csv


In [11]:
for path in sorted(PROCESSED_DIR.glob("disease_embeddings_node2vec*.csv")):
    print(path.name)

disease_embeddings_node2vec_dp.csv
disease_embeddings_node2vec_dph.csv
